################ MD01: NOTEBOOK OVERVIEW ################
# WILLMA Whisper Transcriber V05e

This notebook provides a local-PC batch transcription workflow for audio files stored in a folder.

It is organized as separate sections for:

- setup and notebook usage
- dependency handling
- configuration
- audio discovery and validation
- Whisper model loading
- single-file and batch transcription
- transcript formatting and saving
- progress and error reporting
- end-to-end execution
- preview and verification

The workflow is designed for use in Visual Studio Code with Jupyter support.

**V05e focus:** secrets are no longer stored in the notebook and must be loaded from a local `.whisper-env` file.


################ MD02: V05e IMPROVEMENTS ################
## What's new in V05e

V05e builds on V05d and moves all secret handling and local path configuration out of the notebook into a local `.whisper-env` file.

### Changes in V05e

#### 1 — Secret management via `.whisper-env`
The notebook now reads sensitive values from a `.whisper-env` file stored next to the notebook.
No API key should remain hard-coded in any notebook cell.

Expected keys:

| Key | Required | Purpose |
|---|---|---|
| `WILLMA_API_KEY` | Yes | API key used for WILLMA authentication |
| `WILLMA_BASE_URL` | No | Overrides the default WILLMA base URL |
| `DEFAULT_INPUT_FOLDER` | Yes | Defines the notebook input folder |

Example `.whisper-env` content:

```text
WILLMA_API_KEY=<YOUR_API_KEY>
WILLMA_BASE_URL=https://willma.surf.nl/api/v0
DEFAULT_INPUT_FOLDER=E:\TEMP\TRANSCRIPTIE_JUNI_2026\1046971@hr.nl
```

#### 2 — Safer startup validation
The setup cell now validates whether the env file exists and whether the required API key and input folder were loaded.
It raises a clear error before batch execution if configuration is incomplete.

#### 3 — Documentation refresh
Notebook notes were updated so the setup, security expectations, and integrity workflow are explicit.

#### 4 — Integrity verification retained
The existing output integrity checks remain active for:

- `{stem}.txt`
- `{stem}.segments.json`
- `{stem}.speaker.xlsx`
- `{stem}.speaker.srt`
- `{stem}.transcript.pdf`

It also reports transcript coverage relative to the audio duration.

---
### Inherited from V05d

1. **Styled PDF transcript export** — transcripts are also written as `.pdf` with structured layout.
2. **Excel diarization output** — speaker-aligned diarization output is written as `.xlsx`.
3. **ffmpeg audio extraction for container formats** — container files are converted to a temporary 16 kHz mono WAV before transcription where possible.
4. **Graceful fallback behavior** — transcription continues with alternative handling where supported.
5. **Skip already-processed files** — existing outputs can be detected and reused.
6. **Visible preprocessing progress** — progress messages remain visible during long operations.
7. **Coverage monitoring** — transcript coverage is still checked against source duration.


################ MD03: ENVIRONMENT AND DEPENDENCIES ################
## Environment: `transcriber-env`

This notebook requires the **`transcriber-env`** conda environment.
Activate it in VS Code by selecting the kernel `Python (transcriber-env)` in the top-right
kernel picker, or from the terminal:

```powershell
conda activate transcriber-env
```

---

## Required packages

| Package | Why it is needed |
|---|---|
| **Python 3.11** | Tested runtime for this notebook |
| **pandas** | Tables, summaries, and diarization export |
| **requests** + **urllib3** | HTTP client for WILLMA API calls |
| **soundfile** | Audio metadata and WAV read/write support |
| **scipy** | Filtering and resampling |
| **noisereduce** *(optional)* | Optional noise reduction |
| **openpyxl** | Writes diarization output as `.xlsx` |
| **reportlab** | Writes styled transcript PDFs |
| **ipykernel** | Registers the conda environment as a Jupyter kernel |
| **IPython** | `display()` for notebook tables |

### External tool

| Tool | Why it is needed |
|---|---|
| **ffmpeg** (on system PATH) | Extracts audio from container files before transcription |

### Installing missing packages into `transcriber-env`

```powershell
conda activate transcriber-env
pip install pandas requests soundfile scipy noisereduce ipykernel openpyxl reportlab
python -m ipykernel install --user --name transcriber-env --display-name "Python (transcriber-env)"
```

Verify ffmpeg is reachable from the notebook kernel:

```python
import subprocess
subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode  # should be 0
```

---
## `.whisper-env` requirements

Place a file named `.whisper-env` in the same folder as this notebook.

Required content:

```text
WILLMA_API_KEY=<YOUR_API_KEY>
DEFAULT_INPUT_FOLDER=E:\TEMP\TRANSCRIPTIE_JUNI_2026\1046971@hr.nl
```

Optional keys:

```text
WILLMA_BASE_URL=https://willma.surf.nl/api/v0
```

If the env file is missing, `WILLMA_API_KEY` is empty, or `DEFAULT_INPUT_FOLDER` is missing,
the setup cell will stop with a clear error.


In [ ]:
################ CELL01: SETUP — PATHS, IMPORTS AND CONFIGURATION ################
from pathlib import Path

NOTEBOOK_PATH = Path(
    r"D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\WILLMA_WHISPER_TRANSCRIBER_V05e.ipynb"
)
ENV_FILE = NOTEBOOK_PATH.with_name(".whisper-env")

import io
import json
import mimetypes
import subprocess
import time

import pandas as pd
import requests
import soundfile as sf
from IPython.display import display
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (
    HRFlowable,
    KeepTogether,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)
from requests.adapters import HTTPAdapter
from scipy import signal
from urllib3.util.retry import Retry

try:
    import noisereduce as nr  # type: ignore[reportMissingImports]
except ImportError:
    nr = None


def load_whisper_env(env_path):
    path = Path(env_path)
    if not path.exists():
        raise FileNotFoundError(
            "Missing env file. Create `.whisper-env` next to the notebook "
            "and define WILLMA_API_KEY plus DEFAULT_INPUT_FOLDER."
        )

    env = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env


WHISPER_ENV = load_whisper_env(ENV_FILE)
DEFAULT_INPUT_FOLDER_VALUE = WHISPER_ENV.get("DEFAULT_INPUT_FOLDER", "").strip()
if not DEFAULT_INPUT_FOLDER_VALUE:
    raise ValueError(
        "DEFAULT_INPUT_FOLDER is not configured in `.whisper-env`. "
        "Add a valid input folder before running this notebook."
    )

DEFAULT_INPUT_FOLDER = Path(DEFAULT_INPUT_FOLDER_VALUE).expanduser()
DEFAULT_OUTPUT_FOLDER = DEFAULT_INPUT_FOLDER / "transcriber_v05e_outputs"
DEFAULT_CLEANED_FOLDER = DEFAULT_OUTPUT_FOLDER / "cleaned_audio"
DEFAULT_LOG_FOLDER = DEFAULT_OUTPUT_FOLDER / "logs"
DEFAULT_PROGRESS_FILE = DEFAULT_LOG_FOLDER / "batch_progress.json"
DEFAULT_REFERENCE_FOLDER = DEFAULT_OUTPUT_FOLDER / "references"

for _d in [
    DEFAULT_OUTPUT_FOLDER,
    DEFAULT_CLEANED_FOLDER,
    DEFAULT_LOG_FOLDER,
    DEFAULT_REFERENCE_FOLDER,
]:
    _d.mkdir(parents=True, exist_ok=True)

WILLMA_API_KEY = WHISPER_ENV.get("WILLMA_API_KEY", "").strip()
if not WILLMA_API_KEY or WILLMA_API_KEY == "<YOUR_API_KEY>":
    raise ValueError(
        "WILLMA_API_KEY is not configured in `.whisper-env`. "
        "Add a valid key before running this notebook."
    )


def create_retry_session(max_retries=None):
    n = max_retries or (CONFIG["max_retries"] if "CONFIG" in globals() else 3)
    session = requests.Session()
    retry = Retry(
        total=n,
        connect=n,
        read=n,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "POST"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


CONFIG = {
    "input_folder": DEFAULT_INPUT_FOLDER,
    "output_folder": DEFAULT_OUTPUT_FOLDER,
    "cleaned_folder": DEFAULT_CLEANED_FOLDER,
    "reference_folder": DEFAULT_REFERENCE_FOLDER,
    "progress_file": DEFAULT_PROGRESS_FILE,
    "env_file": ENV_FILE,
    "willma_base_url": (
        WHISPER_ENV.get(
            "WILLMA_BASE_URL",
            "https://willma.surf.nl/api/v0",
        ).strip()
        or "https://willma.surf.nl/api/v0"
    ),
    "willma_api_key": WILLMA_API_KEY,
    "preferred_whisper_names": ["whisper-large-v3", "whisper"],
    "selected_whisper_model_name": None,
    "diarization_model": "pyannote/speaker-diarization",
    "supported_extensions": {
        ".3g2", ".3gp", ".aac", ".aif", ".aiff", ".alac", ".amr", ".ape",
        ".au", ".avi", ".caf", ".dts", ".eac3", ".f4a", ".f4b", ".flac",
        ".flv", ".m1a", ".m2a", ".m4a", ".m4b", ".m4p", ".m4v", ".mka",
        ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpa", ".mpc", ".mpeg",
        ".mpg", ".oga", ".ogg", ".ogm", ".ogv", ".opus", ".ra", ".rm",
        ".spx", ".ts", ".wav", ".weba", ".webm", ".wma", ".wmv",
    },
    "language": "nl",
    "request_timeout": 1800,
    "max_retries": 3,
    "sleep_between_attempts": 6,
    "max_diarization_attempts": 2,
    "max_audio_mb": 24,
    "overwrite_outputs": False,
    "preprocess_audio": False,
    "prefer_cleaned_audio": True,
    "target_sample_rate": 16000,
    "highpass_hz": 80,
    "noise_reduction_strength": 0.6,
    "peak_normalization_target": 0.98,
    "use_api_diarization": True,
    "alternative_pause_threshold": 1.2,
    "min_overlap_seconds": 0.15,
}

CONTAINER_EXTENSIONS = {
    ".3g2", ".3gp", ".aac", ".aif", ".aiff", ".alac", ".amr", ".ape", ".au",
    ".avi", ".caf", ".dts", ".eac3", ".f4a", ".f4b", ".flac", ".flv", ".m1a",
    ".m2a", ".m4a", ".m4b", ".m4p", ".m4v", ".mka", ".mkv", ".mov", ".mp2",
    ".mp3", ".mp4", ".mpa", ".mpc", ".mpeg", ".mpg", ".oga", ".ogg", ".ogm",
    ".ogv", ".opus", ".ra", ".rm", ".spx", ".ts", ".weba", ".webm", ".wma",
    ".wmv",
}
DIRECT_AUDIO_EXTENSIONS = {".wav"}

print(f"Using env file: {ENV_FILE}")
print(f"Input folder: {CONFIG['input_folder']}")
print(f"Output folder: {CONFIG['output_folder']}")
print(f"WILLMA base URL: {CONFIG['willma_base_url']}")
print("WILLMA API key loaded from env file.")


In [38]:
################ CELL02: OPTIONAL PACKAGE INSTALLATION ################
# Optional install cell
# Uncomment if packages are missing in the active environment.

# %pip install pandas requests soundfile scipy openai-whisper noisereduce openpyxl reportlab


In [ ]:
################ CELL03: ALL FUNCTIONS ################

# ── Path and audio discovery ──────────────────────────────────────────────────────
def normalize_path(p):
    return Path(p).expanduser().resolve()


def log_message(message, level="INFO"):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{level}] {message}")
    try:
        import sys
        sys.stdout.flush()
    except Exception:
        pass


def is_valid_media_file(path):
    p = normalize_path(path)
    if not p.exists() or not p.is_file() or p.stat().st_size <= 0:
        return False
    ext = p.suffix.lower()
    if ext in DIRECT_AUDIO_EXTENSIONS:
        try:
            sf.info(str(p))
            return True
        except Exception:
            return False
    if ext not in CONFIG["supported_extensions"]:
        return False
    probe_cmd = [
        "ffmpeg",
        "-v",
        "error",
        "-i",
        str(p),
        "-map",
        "0:a:0",
        "-f",
        "null",
        "-",
    ]
    try:
        completed = subprocess.run(probe_cmd, capture_output=True, text=True, check=False)
        return completed.returncode == 0
    except FileNotFoundError:
        return True


def find_audio_files(folder_path, supported_extensions=None):
    folder = normalize_path(folder_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    if not folder.exists():
        return []
    candidates = sorted(
        p for p in folder.glob("*")
        if p.is_file() and p.suffix.lower() in exts
    )
    return [p for p in candidates if is_valid_media_file(p)]


def prepare_batch_file_list(folder_path, supported_extensions=None):
    return pd.DataFrame(
        [
            {"file_path": str(p), "file_name": p.name, "extension": p.suffix.lower()}
            for p in find_audio_files(folder_path, supported_extensions)
        ]
    )


# ── Audio validation ──────────────────────────────────────────────────────────────
def validate_audio_file(file_path, supported_extensions=None):
    path = normalize_path(file_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    result = {
        "file_path": str(path),
        "exists": path.exists(),
        "is_file": path.is_file(),
        "supported_extension": path.suffix.lower() in exts,
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else 0,
        "readable": False,
        "valid": False,
        "message": "",
    }
    if not result["exists"]:
        result["message"] = "File does not exist."
        return result
    if not result["is_file"]:
        result["message"] = "Path is not a file."
        return result
    if not result["supported_extension"]:
        result["message"] = "Unsupported media extension."
        return result
    if result["size_bytes"] <= 0:
        result["message"] = "File is empty."
        return result
    if path.suffix.lower() in DIRECT_AUDIO_EXTENSIONS:
        try:
            sf.info(str(path))
            result["readable"] = True
            result["valid"] = True
            result["message"] = "OK"
        except Exception as exc:
            result["message"] = f"Audio read failed: {exc}"
        return result

    result["readable"] = is_valid_media_file(path)
    result["valid"] = result["readable"]
    result["message"] = "OK (ffmpeg-readable media)" if result["valid"] else "Media is not readable by ffmpeg."
    return result


def run_preflight_checks(file_paths):
    return pd.DataFrame([validate_audio_file(p) for p in file_paths])


# ── Model catalog ─────────────────────────────────────────────────────────────────
def find_model_by_name_fragment(items, fragments):
    for frag in [(f or "").lower() for f in (fragments or [])]:
        for item in items:
            if frag in str(item.get("name", "")).lower():
                return item
    return None


def load_model_catalog(session=None):
    resp = (session or create_retry_session()).get(
        f"{CONFIG['willma_base_url']}/sequences",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


def load_whisper_model(session=None, preferred_names=None):
    catalog = load_model_catalog(session=session)
    sel = find_model_by_name_fragment(catalog, preferred_names or CONFIG["preferred_whisper_names"])
    if sel is None:
        sel = next((x for x in catalog if "whisper" in str(x.get("name", "")).lower()), None)
    if sel is None:
        raise ValueError("No Whisper model found in the WILLMA model catalog.")
    CONFIG["selected_whisper_model_name"] = sel.get("name")
    return sel


# ── Transcription helpers ─────────────────────────────────────────────────────────
def load_audio_metadata(file_path):
    path = normalize_path(file_path)
    if path.suffix.lower() in CONTAINER_EXTENSIONS:
        size_mb = path.stat().st_size / 1024 / 1024
        return {
            "file_path": str(path),
            "samplerate": None,
            "channels": None,
            "duration_seconds": None,
            "format": path.suffix.lstrip(".").upper(),
            "subtype": "container",
            "size_mb": round(size_mb, 2),
        }
    info = sf.info(str(path))
    return {
        "file_path": str(path),
        "samplerate": info.samplerate,
        "channels": info.channels,
        "duration_seconds": round(info.duration, 3),
        "format": info.format,
        "subtype": info.subtype,
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
    }


# ── ffmpeg audio extraction ───────────────────────────────────────────────────────
def extract_audio_from_container(src_path, out_dir=None):
    src = normalize_path(src_path)
    dst_dir = normalize_path(out_dir or CONFIG["cleaned_folder"])
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{src.stem}.extracted.wav"

    if dst.exists() and not CONFIG["overwrite_outputs"]:
        log_message(f"{src.name}: extracted WAV already exists — reusing {dst.name}")
        return dst

    cmd = [
        "ffmpeg", "-y", "-i", str(src), "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(dst),
    ]

    try:
        log_message(f"{src.name}: extracting audio via ffmpeg → {dst.name} ({src.stat().st_size / 1024 / 1024:.1f} MB input)")
        create_no_window = getattr(subprocess, "CREATE_NO_WINDOW", 0)
        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            creationflags=create_no_window,
        )
        if proc.returncode != 0:
            err = proc.stderr.decode(errors="replace")[-600:]
            log_message(f"{src.name}: ffmpeg failed (rc={proc.returncode}): {err}", level="WARNING")
            return None
        if not dst.exists() or dst.stat().st_size == 0:
            log_message(f"{src.name}: ffmpeg did not create a valid extracted WAV file", level="WARNING")
            return None
        out_mb = dst.stat().st_size / 1024 / 1024
        log_message(f"{src.name}: extraction done — {dst.name} ({out_mb:.1f} MB)")
        return dst
    except FileNotFoundError:
        log_message(f"{src.name}: ffmpeg not found on PATH — falling back to raw-bytes upload", level="WARNING")
        return None
    except Exception as exc:
        log_message(f"{src.name}: ffmpeg error: {exc}", level="WARNING")
        return None


def to_mono(audio):
    return audio.astype("float32") if getattr(audio, "ndim", 1) == 1 else audio.mean(axis=1).astype("float32")


def resample_audio_array(audio, src_sr, target_sr):
    if src_sr == target_sr:
        return audio
    duration = len(audio) / src_sr if src_sr else 0
    target_len = max(int(duration * target_sr), 1)
    return signal.resample(audio, target_len)


def ensure_mono(audio):
    if getattr(audio, "ndim", 1) == 1:
        return audio
    return audio.mean(axis=1)


def preprocess_audio(file_path, out_dir=None, config=None):
    cfg = config or CONFIG
    src = normalize_path(file_path)
    if src.suffix.lower() in CONTAINER_EXTENSIONS or src.stem.endswith(".extracted"):
        log_message(f"{src.name}: skipping DSP preprocessing (container or extracted WAV)")
        return None
    out_root = normalize_path(out_dir or cfg["cleaned_folder"])
    out_root.mkdir(parents=True, exist_ok=True)
    dst = out_root / f"{src.stem}.cleaned.wav"
    if dst.exists() and not cfg.get("overwrite_outputs", False):
        log_message(f"{src.name}: cleaned file already exists — skipping preprocessing")
        return dst

    mb = src.stat().st_size / 1024 / 1024
    log_message(f"{src.name}: reading audio ({mb:.1f} MB)")
    audio, sr = sf.read(str(src), always_2d=False)
    audio = to_mono(audio)
    dur = len(audio) / sr
    log_message(f"{src.name}: {dur/60:.1f} min, {sr} Hz — starting preprocessing")

    target_sr = int(cfg.get("target_sample_rate", 16000))
    if sr != target_sr:
        log_message(f"{src.name}: resampling {sr} Hz → {target_sr} Hz (this may take a while)")
        audio = signal.resample(audio, int(round(len(audio) * target_sr / sr))).astype("float32")
        sr = target_sr

    log_message(f"{src.name}: applying high-pass filter")
    b, a = signal.butter(4, min(cfg["highpass_hz"] / (sr / 2), 0.99), btype="highpass")
    audio = signal.filtfilt(b, a, audio).astype("float32")

    if nr is not None:
        log_message(f"{src.name}: reducing noise (this can take several minutes for long files)")
        noise = audio[: min(len(audio), int(sr * 1.5))]
        audio = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise,
            prop_decrease=cfg["noise_reduction_strength"],
            stationary=True,
        ).astype("float32")

    peak = float(abs(audio).max()) if len(audio) else 0.0
    if peak > 0:
        audio = (audio / peak) * float(cfg.get("peak_normalization_target", 0.98))

    log_message(f"{src.name}: writing cleaned file → {dst.name}")
    sf.write(str(dst), audio, sr, subtype="PCM_16")
    log_message(f"{src.name}: preprocessing done")
    return dst


def choose_transcription_source(original_path, cleaned_path=None):
    orig = normalize_path(original_path)
    cln = normalize_path(cleaned_path) if cleaned_path else orig.with_name(f"{orig.stem}.cleaned.wav")
    return cln if CONFIG["prefer_cleaned_audio"] and cln.exists() else orig


def _transcribe_bytes(audio_bytes, filename, session, language):
    mime = mimetypes.guess_type(filename)[0] or "application/octet-stream"
    resp = (session or create_retry_session()).post(
        f"{CONFIG['willma_base_url']}/audio/transcriptions",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Accept": "application/json"},
        files={"file": (filename, audio_bytes, mime)},
        data={
            "model": CONFIG["selected_whisper_model_name"],
            "language": language or CONFIG["language"],
            "stream": "false",
            "response_format": "verbose_json",
            "timestamp_granularities[]": "segment",
        },
        timeout=CONFIG["request_timeout"],
    )
    resp.raise_for_status()
    result = resp.json()
    if isinstance(result, dict) and result.get("error"):
        raise RuntimeError(f"WILLMA API error: {result['error']}")
    return result


def _chunk_wav_and_transcribe(path, session, language):
    import math

    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)
    audio, sr = sf.read(str(path), always_2d=False)
    audio = to_mono(audio)
    bytes_per_sec = sr * 2
    chunk_secs = max(60, max_bytes // max(bytes_per_sec, 1))
    chunk_samples = chunk_secs * sr
    n_chunks = math.ceil(len(audio) / chunk_samples)

    all_segments, text_parts = [], []
    for i in range(n_chunks):
        s0 = i * chunk_samples
        s1 = min(s0 + chunk_samples, len(audio))
        offset = s0 / sr
        buf = io.BytesIO()
        sf.write(buf, audio[s0:s1], sr, subtype="PCM_16", format="WAV")
        log_message(f"{path.name}: chunk {i+1}/{n_chunks} ({offset:.1f}s–{s1/sr:.1f}s, {buf.tell()/1024/1024:.1f} MB)")
        r = _transcribe_bytes(buf.getvalue(), path.name, session, language)
        text_parts.append(r.get("text", "").strip())
        for seg in r.get("segments", []):
            adj = dict(seg)
            adj["start"] = round(float(seg.get("start", 0)) + offset, 3)
            adj["end"] = round(float(seg.get("end", 0)) + offset, 3)
            all_segments.append(adj)

    return {
        "file_path": str(path),
        "metadata": load_audio_metadata(path),
        "detected_language": language or CONFIG["language"],
        "text": " ".join(t for t in text_parts if t),
        "segments": all_segments,
        "raw_result": {"chunked": True, "n_chunks": n_chunks},
    }


def overlap_seconds(s_a, e_a, s_b, e_b):
    return max(0.0, min(e_a, e_b) - max(s_a, s_b))


def request_diarization_with_retries(
    session, base_url, api_key, model_name, filename, audio_bytes, mime_type, timeout_value, max_attempts, sleep_secs,
):
    last_resp, last_result = None, {}
    for attempt in range(1, max_attempts + 1):
        try:
            last_resp = session.post(
                f"{base_url}/audio/custom-diarization",
                headers={"X-API-KEY": api_key, "Accept": "application/json"},
                data={"model": model_name},
                files={"file": (filename, audio_bytes, mime_type)},
                timeout=timeout_value,
            )
            try:
                last_result = last_resp.json()
            except ValueError:
                last_result = last_resp.text
            if last_resp.status_code < 400 and isinstance(last_result, dict) and not last_result.get("error"):
                return last_resp, last_result
        except Exception as exc:
            last_resp = None
            last_result = {"error": f"Attempt {attempt}: {exc}"}
        if attempt < max_attempts:
            time.sleep(sleep_secs)
    return last_resp, last_result


def build_turns_from_stt(segments, pause_threshold=1.2):
    if not segments:
        return []
    segs = sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0))))
    cur = {"start": float(segs[0].get("start", 0)), "end": float(segs[0].get("end", 0)), "text": (segs[0].get("text") or "").strip()}
    turns = []
    for seg in segs[1:]:
        s, e, t = float(seg.get("start", 0)), float(seg.get("end", 0)), (seg.get("text") or "").strip()
        if s - cur["end"] >= pause_threshold:
            turns.append(cur)
            cur = {"start": s, "end": e, "text": t}
        else:
            cur["end"] = max(cur["end"], e)
            if t:
                cur["text"] = (cur["text"] + " " + t).strip()
    turns.append(cur)
    return turns


def alternative_diarization_from_stt(segments, pause_threshold=1.2, start_with="001"):
    cycle = ["002", "001"] if start_with == "002" else ["001", "002"]
    return [
        {"start": round(float(t["start"]), 3), "end": round(float(t["end"]), 3), "speaker": cycle[i % 2]}
        for i, t in enumerate(build_turns_from_stt(segments, pause_threshold))
    ]


def force_two_speaker_labels(segments):
    counts = {}
    for seg in segments:
        key = str(seg.get("speaker", "")).strip()
        if key and key != "UNKNOWN":
            counts[key] = counts.get(key, 0) + 1
    top = [x[0] for x in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:2]]
    mapping = {top[i]: f"{i+1:03d}" for i in range(len(top))}
    forced = []
    for seg in segments:
        upd = dict(seg)
        key = str(upd.get("speaker", "")).strip()
        upd["speaker"] = mapping.get(key, key if key in ("001", "002") else "UNKNOWN") if key and key != "UNKNOWN" else "UNKNOWN"
        forced.append(upd)
    return forced


def assign_unknown_by_neighbors(segments):
    s = [dict(x) for x in segments]
    for i, seg in enumerate(s):
        if seg.get("speaker") != "UNKNOWN":
            continue
        prev = next((s[j]["speaker"] for j in range(i - 1, -1, -1) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        nxt = next((s[j]["speaker"] for j in range(i + 1, len(s)) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        if prev and prev == nxt:
            seg["speaker"] = prev
    return s


def merge_speaker_segments(segments, max_gap=1.0):
    merged = []
    for seg in sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0)))):
        text = seg.get("text", "").strip()
        if not text:
            continue
        if not merged:
            merged.append(dict(seg))
            continue
        prev = merged[-1]
        if prev.get("speaker") == seg.get("speaker") and float(seg.get("start", 0)) - float(prev.get("end", 0)) <= max_gap:
            prev["end"] = seg.get("end", prev.get("end"))
            prev["text"] = (prev.get("text", "").strip() + " " + text).strip()
        else:
            merged.append(dict(seg))
    return merged


def diarize_audio_or_segments(file_path, transcript_segments, session=None):
    path = normalize_path(file_path)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    _, result = request_diarization_with_retries(
        session or create_retry_session(),
        CONFIG["willma_base_url"],
        CONFIG["willma_api_key"],
        CONFIG["diarization_model"],
        path.name,
        path.read_bytes(),
        mime,
        CONFIG["request_timeout"],
        CONFIG["max_diarization_attempts"],
        CONFIG["sleep_between_attempts"],
    )
    if isinstance(result, dict) and isinstance(result.get("diarization"), list) and result["diarization"]:
        return result["diarization"]
    return alternative_diarization_from_stt(transcript_segments, pause_threshold=CONFIG["alternative_pause_threshold"])


def align_speakers_to_transcript(transcript_segments, diarization_segments):
    aligned = []
    for seg in transcript_segments or []:
        upd = dict(seg)
        s, e = float(upd.get("start", 0)), float(upd.get("end", 0))
        best_sp, best_ov = "UNKNOWN", 0.0
        for d in diarization_segments or []:
            ov = overlap_seconds(s, e, float(d.get("start", 0)), float(d.get("end", 0)))
            if ov > best_ov:
                best_ov = ov
                best_sp = d.get("speaker") or d.get("label") or d.get("id") or "UNKNOWN"
        upd["speaker"] = best_sp if best_ov >= CONFIG["min_overlap_seconds"] else "UNKNOWN"
        upd["speaker_overlap"] = round(best_ov, 3)
        upd["speaker_source"] = "api_or_fallback"
        aligned.append(upd)
    return merge_speaker_segments(assign_unknown_by_neighbors(force_two_speaker_labels(aligned)))


def transcribe_single_file(file_path, session=None, language=None):
    path = normalize_path(file_path)
    s = session or create_retry_session()
    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)

    if path.suffix.lower() in CONTAINER_EXTENSIONS:
        log_message(f"{path.name}: container fallback — sending raw bytes to API (no chunking)")
        with path.open("rb") as f:
            audio_bytes = f.read()
        r = _transcribe_bytes(audio_bytes, path.name, s, language)
        return {
            "file_path": str(path),
            "metadata": load_audio_metadata(path),
            "detected_language": r.get("language", language or CONFIG["language"]),
            "text": r.get("text", "").strip(),
            "segments": r.get("segments", []),
            "raw_result": r,
        }

    if path.stat().st_size > max_bytes:
        log_message(f"{path.name} is {path.stat().st_size/1024/1024:.1f} MB — splitting into chunks")
        return _chunk_wav_and_transcribe(path, s, language)

    with path.open("rb") as f:
        audio_bytes = f.read()
    r = _transcribe_bytes(audio_bytes, path.name, s, language)
    return {
        "file_path": str(path),
        "metadata": load_audio_metadata(path),
        "detected_language": r.get("language", language or CONFIG["language"]),
        "text": r.get("text", "").strip(),
        "segments": r.get("segments", []),
        "raw_result": r,
    }


def postprocess_transcript(text):
    c = " ".join(str(text or "").split())
    return (c[0].upper() + c[1:]) if c and c[0].isalpha() else c


def levenshtein_distance(a, b):
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for ta in a:
        row = [prev[0] + 1]
        for j, tb in enumerate(b, 1):
            row.append(min(row[-1] + 1, prev[j] + 1, prev[j - 1] + (ta != tb)))
        prev = row
    return prev[-1]


def evaluate_transcript(reference_text, predicted_text):
    ref = " ".join(str(reference_text or "").strip().lower().split())
    pred = " ".join(str(predicted_text or "").strip().lower().split())
    rw, pw = ref.split(), pred.split()
    return {
        "wer": 0.0 if not rw and not pw else levenshtein_distance(rw, pw) / max(len(rw), 1),
        "cer": 0.0 if not ref and not pred else levenshtein_distance(list(ref), list(pred)) / max(len(ref), 1),
    }


def format_srt_timestamp(secs: float) -> str:
    ms = int(round(secs * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def prepare_output_paths(file_path, output_folder=None):
    stem = normalize_path(file_path).stem
    out = normalize_path(output_folder or CONFIG["output_folder"])
    out.mkdir(parents=True, exist_ok=True)
    return {
        "txt": out / f"{stem}.txt",
        "json": out / f"{stem}.segments.json",
        "csv": out / f"{stem}.speaker.csv",
        "xlsx": out / f"{stem}.speaker.xlsx",
        "srt": out / f"{stem}.speaker.srt",
        "pdf": out / f"{stem}.transcript.pdf",
    }


def format_segments_as_table(segments):
    return pd.DataFrame(segments) if segments else pd.DataFrame(columns=["start", "end", "speaker", "text", "speaker_source", "speaker_overlap"])


def write_txt_transcript(p, text):
    Path(p).write_text(str(text or "").strip() + "\n", encoding="utf-8")


def write_json(p, payload):
    Path(p).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_csv(p, segments):
    format_segments_as_table(segments).to_csv(p, index=False, encoding="utf-8-sig")


def write_srt(output_path, segments):
    blocks = []
    idx = 1
    for seg in segments:
        text = str(seg.get("text", "")).strip()
        if not text:
            continue
        blocks.append(
            f"{idx}\n{format_srt_timestamp(float(seg.get('start', 0)))} --> {format_srt_timestamp(float(seg.get('end', 0)))}\n[{seg.get('speaker', 'UNKNOWN')}] {text}\n"
        )
        idx += 1
    Path(output_path).write_text("\n".join(blocks), encoding="utf-8")


def write_pdf_transcript(output_path, transcript_text, segments=None, metadata=None, source_file=None):
    doc = SimpleDocTemplate(str(output_path), pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm, topMargin=16 * mm, bottomMargin=16 * mm)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("TranscriptTitle", parent=styles["Heading1"], fontName="Helvetica-Bold", fontSize=15, leading=18, textColor=colors.HexColor("#1f2937"), spaceAfter=8)
    meta_style = ParagraphStyle("TranscriptMeta", parent=styles["BodyText"], fontName="Helvetica", fontSize=8.5, leading=11, textColor=colors.HexColor("#4b5563"))
    speaker_style = ParagraphStyle("TranscriptSpeaker", parent=styles["BodyText"], fontName="Helvetica-Bold", fontSize=9.5, leading=12, textColor=colors.HexColor("#111827"), spaceAfter=1)
    text_style = ParagraphStyle("TranscriptText", parent=styles["BodyText"], fontName="Helvetica", fontSize=10, leading=14, textColor=colors.HexColor("#111827"), alignment=TA_LEFT)
    story = [Paragraph("Transcript", title_style)]

    meta_rows = [["Source file", Path(source_file).name if source_file else ""], ["Format", str((metadata or {}).get("format", ""))], ["Duration (s)", str((metadata or {}).get("duration_seconds", ""))], ["Generated", time.strftime("%Y-%m-%d %H:%M:%S")]]
    meta_table = Table(meta_rows, colWidths=[35 * mm, 120 * mm])
    meta_table.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#f9fafb")), ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#d1d5db")), ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#e5e7eb")), ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"), ("FONTNAME", (1, 0), (1, -1), "Helvetica"), ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("LEADING", (0, 0), (-1, -1), 10), ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor("#374151")), ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    story.extend([meta_table, Spacer(1, 5 * mm), HRFlowable(width="100%", thickness=0.7, color=colors.HexColor("#d1d5db")), Spacer(1, 4 * mm)])

    diarization_segments = segments or []
    if diarization_segments:
        for seg in diarization_segments:
            speaker = str(seg.get("speaker", "UNKNOWN"))
            start = float(seg.get("start", 0.0) or 0.0)
            end = float(seg.get("end", start) or start)
            text = str(seg.get("text", "")).strip() or ""
            speaker_line = f"{speaker}  |  {format_srt_timestamp(start).replace(',', '.')} - {format_srt_timestamp(end).replace(',', '.')}"
            block = KeepTogether([Paragraph(speaker_line, speaker_style), Paragraph(text.replace("\n", "<br/>"), text_style), Spacer(1, 3 * mm)])
            story.append(block)
    else:
        story.append(Paragraph(str(transcript_text or "").strip().replace("\n", "<br/>"), text_style))

    doc.build(story)


def save_transcription_outputs(result, output_folder=None):
    paths = prepare_output_paths(result["file_path"], output_folder=output_folder)
    write_txt_transcript(paths["txt"], result.get("text", ""))
    write_json(paths["json"], result)
    write_csv(paths["csv"], result.get("segments", []))
    format_segments_as_table(result.get("segments", [])).to_excel(paths["xlsx"], index=False)
    write_srt(paths["srt"], result.get("segments", []))
    write_pdf_transcript(paths["pdf"], result.get("text", ""), segments=result.get("segments", []), metadata=result.get("metadata"), source_file=result.get("file_path"))
    return paths


def is_already_processed(file_path, settings=None):
    settings = settings or CONFIG
    src = normalize_path(file_path)
    out = normalize_path(settings.get("output_folder") or CONFIG["output_folder"])

    def _outputs_exist(stem):
        return all((out / fname).exists() for fname in (f"{stem}.txt", f"{stem}.segments.json", f"{stem}.speaker.csv", f"{stem}.speaker.xlsx", f"{stem}.speaker.srt", f"{stem}.transcript.pdf"))

    return _outputs_exist(src.stem) or _outputs_exist(f"{src.stem}.cleaned") or _outputs_exist(f"{src.stem}.extracted")


def verify_output_integrity(file_path, output_paths, transcript_text="", metadata=None):
    meta = metadata or {}
    duration_seconds = meta.get("duration_seconds")
    transcript_chars = len((transcript_text or "").strip())
    rows = {
        "source_file": str(file_path),
        "txt_exists": bool(output_paths.get("txt") and Path(output_paths["txt"]).exists()),
        "json_exists": bool(output_paths.get("json") and Path(output_paths["json"]).exists()),
        "csv_exists": bool(output_paths.get("csv") and Path(output_paths["csv"]).exists()),
        "xlsx_exists": bool(output_paths.get("xlsx") and Path(output_paths["xlsx"]).exists()),
        "srt_exists": bool(output_paths.get("srt") and Path(output_paths["srt"]).exists()),
        "pdf_exists": bool(output_paths.get("pdf") and Path(output_paths["pdf"]).exists()),
        "transcript_chars": transcript_chars,
        "duration_seconds": duration_seconds,
        "coverage_chars_per_second": round(transcript_chars / duration_seconds, 3) if duration_seconds else None,
    }
    rows["ok"] = all(rows[k] for k in ["txt_exists", "json_exists", "csv_exists", "xlsx_exists", "srt_exists", "pdf_exists"]) and transcript_chars > 0
    return rows


def transcribe_with_willma(file_path, session=None, settings=None):
    cfg = settings or CONFIG
    src = normalize_path(file_path)
    if src.suffix.lower() in CONTAINER_EXTENSIONS:
        extracted = extract_audio_from_container(src, out_dir=cfg.get("cleaned_folder") or CONFIG["cleaned_folder"])
        if extracted and extracted.exists():
            tx_src = extracted
            log_message(f"{src.name}: using extracted WAV → {extracted.name}")
        else:
            tx_src = src
            log_message(f"{src.name}: ffmpeg extraction failed, uploading raw container bytes", level="WARNING")
    else:
        cleaned = None
        if cfg.get("preprocess_audio"):
            try:
                cleaned = preprocess_audio(src, out_dir=cfg.get("cleaned_folder"), config=cfg)
            except Exception as exc:
                log_message(f"Preprocessing failed for {src.name}: {exc}", level="WARNING")
        tx_src = choose_transcription_source(src, cleaned)

    result = transcribe_single_file(tx_src, session=session, language=cfg.get("language"))
    segs = result.get("segments") or []
    if segs:
        merged = align_speakers_to_transcript(segs, diarize_audio_or_segments(tx_src, segs, session=session or create_retry_session()))
        if merged:
            result["text"] = postprocess_transcript(" ".join(x.get("text", "").strip() for x in merged if x.get("text")).strip() or result.get("text", ""))
            result["segments"] = merged

    if not result.get("segments"):
        fb = postprocess_transcript(result.get("text", ""))
        result["text"] = fb
        result["segments"] = [{"start": 0.0, "end": float((result.get("metadata") or {}).get("duration_seconds", 0.0) or 0.0), "speaker": "001", "speaker_source": "transcript_fallback", "speaker_overlap": 0.0, "text": fb}] if fb else []

    ref = normalize_path(cfg["reference_folder"]) / f"{src.stem}.reference.txt"
    result["evaluation"] = evaluate_transcript(ref.read_text(encoding="utf-8"), result["text"]) if ref.exists() else None
    result.update({"status": "ok", "source_audio": str(src), "cleaned_audio": str(tx_src) if tx_src != src else "", "transcription_audio": str(tx_src)})
    result["file_path"] = str(src)
    return result


def process_single_audio_file(file_path, session=None, settings=None):
    cfg = settings or CONFIG
    s = session or create_retry_session()
    v = validate_audio_file(file_path, supported_extensions=cfg["supported_extensions"])
    if not v["valid"]:
        return {"file_path": str(file_path), "status": "failed", "error": v["message"], "integrity": {"ok": False}}
    src = normalize_path(file_path)

    if not cfg.get("overwrite_outputs") and is_already_processed(src, cfg):
        out_paths = prepare_output_paths(src, output_folder=cfg.get("output_folder"))
        log_message(f"{src.name}: already processed — skipping (set overwrite_outputs=True to reprocess)")
        transcript_text = Path(out_paths["txt"]).read_text(encoding="utf-8") if Path(out_paths["txt"]).exists() else ""
        integrity = verify_output_integrity(src, {k: str(v) for k, v in out_paths.items()}, transcript_text=transcript_text, metadata=load_audio_metadata(src))
        return {"file_path": str(src), "status": "skipped", "transcription_audio": str(src), "output_paths": {k: str(v) for k, v in out_paths.items()}, "integrity": integrity}

    result = transcribe_with_willma(src, session=s, settings=cfg)
    output_paths = {k: str(v) for k, v in save_transcription_outputs(result, output_folder=cfg["output_folder"]).items()}
    result["output_paths"] = output_paths
    result["integrity"] = verify_output_integrity(src, output_paths, transcript_text=result.get("text", ""), metadata=result.get("metadata"))
    return result


def summarize_batch_results(results):
    rows = []
    for item in results:
        integrity = item.get("integrity") or {}
        rows.append({
            "file_path": item.get("file_path", ""),
            "status": item.get("status", "unknown"),
            "detected_language": item.get("detected_language", ""),
            "model_name": CONFIG.get("selected_whisper_model_name", ""),
            "text_length": len(item.get("text", "") or ""),
            "segment_count": len(item.get("segments", []) or []),
            "transcription_audio": item.get("transcription_audio", ""),
            "txt_exists": integrity.get("txt_exists", False),
            "json_exists": integrity.get("json_exists", False),
            "csv_exists": integrity.get("csv_exists", False),
            "xlsx_exists": integrity.get("xlsx_exists", False),
            "srt_exists": integrity.get("srt_exists", False),
            "pdf_exists": integrity.get("pdf_exists", False),
            "integrity_ok": integrity.get("ok", False),
            "error": item.get("error", ""),
        })
    return pd.DataFrame(rows)


def resume_incomplete_runs(progress_file):
    p = Path(progress_file)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else []


def save_progress_log(progress_file, results):
    p = Path(progress_file)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")


def process_audio_folder(folder_path, session=None, settings=None):
    cfg = settings or CONFIG
    s = session or create_retry_session()
    results = []
    for path in find_audio_files(folder_path, supported_extensions=cfg["supported_extensions"]):
        log_message(f"Processing {path.name}")
        try:
            results.append(process_single_audio_file(path, session=s, settings=cfg))
        except Exception as exc:
            results.append({"file_path": str(path), "status": "failed", "error": str(exc), "integrity": {"ok": False}})
        save_progress_log(cfg["progress_file"], results)
    return results


In [40]:
# ── Diagnostic: list all discovered audio files ───────────────────────────────────
found = find_audio_files(CONFIG["input_folder"])
print(f"Found {len(found)} file(s):")
for p in found:
    print(" ", p)


Found 6 file(s):
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.mp4
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\Interview - ervaringen LAge Rug Support-20240902_155757-Opname van vergadering.mp4
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\Interview LAge Rug Support - Hogeschool Rotterdam-20240424_144746-Opname van vergadering.mp4
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\LARS interview-20240311_100442-Opname van vergadering.mp4
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\MTS78-00KV 120224.mp4
  E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\RG95-00kv011223.mp4


In [ ]:
################ CELL05: BATCH RUN / DEMO ################

session = create_retry_session()

if not str(CONFIG["willma_api_key"]).strip() or CONFIG["willma_api_key"] == "<YOUR_API_KEY>":
    raise ValueError(
        "CONFIG['willma_api_key'] is not set. Add WILLMA_API_KEY to `.whisper-env` before running the batch cell."
    )

try:
    selected_model = load_whisper_model(session=session)
    print(f"Selected model: {selected_model['name']}")
except requests.HTTPError as exc:
    status_code = getattr(exc.response, "status_code", None)
    if status_code == 401:
        raise ValueError(
            "WILLMA authentication failed (401 Unauthorized). Check whether WILLMA_API_KEY in `.whisper-env` contains a valid active API key."
        ) from exc
    raise

audio_file_df = prepare_batch_file_list(CONFIG["input_folder"])
display(audio_file_df)

if audio_file_df.empty or "file_path" not in audio_file_df.columns:
    display(pd.DataFrame([{
        "status": "no_audio_files_found",
        "input_folder": str(CONFIG["input_folder"]),
        "message": "No supported audio files were found, so no transcription or integrity check was run.",
    }]))
    batch_results = []
    verification_rows = []
else:
    valid_files = [str(p) for p in audio_file_df["file_path"].tolist()]
    preflight_df = run_preflight_checks(valid_files)
    display(preflight_df)

    print(f"Processing {len(valid_files)} file(s)...")
    forced_settings = dict(CONFIG)
    forced_settings["overwrite_outputs"] = True
    batch_results = []

    for fp in valid_files:
        try:
            batch_results.append(process_single_audio_file(fp, session=session, settings=forced_settings))
        except Exception as exc:
            batch_results.append({
                "file_path": fp,
                "status": "failed",
                "error": str(exc),
                "output_paths": {},
                "integrity": {"ok": False},
            })
            print(f"FAILED: {Path(fp).name} -> {exc}")

    display(summarize_batch_results(batch_results))

    verification_rows = []
    for item in batch_results:
        out = item.get("output_paths") or {}
        txt_path = Path(out.get("txt", "")) if out.get("txt") else None
        json_path = Path(out.get("json", "")) if out.get("json") else None
        xlsx_path = Path(out.get("xlsx", "")) if out.get("xlsx") else None
        srt_path = Path(out.get("srt", "")) if out.get("srt") else None
        pdf_path = Path(out.get("pdf", "")) if out.get("pdf") else None
        integrity_ok = bool((item.get("integrity") or {}).get("ok", False))
        verification_rows.append({
            "file": Path(item.get("file_path", "")).name,
            "status": item.get("status", ""),
            "error": item.get("error", ""),
            "txt": str(txt_path) if txt_path else "",
            "txt_exists": bool(txt_path and txt_path.exists()),
            "json_exists": bool(json_path and json_path.exists()),
            "xlsx_exists": bool(xlsx_path and xlsx_path.exists()),
            "srt_exists": bool(srt_path and srt_path.exists()),
            "pdf_exists": bool(pdf_path and pdf_path.exists()),
            "integrity_ok": integrity_ok,
            "txt_bytes": txt_path.stat().st_size if txt_path and txt_path.exists() else 0,
            "preview": txt_path.read_text(encoding="utf-8")[:120] if txt_path and txt_path.exists() else "",
        })

if verification_rows:
    display(pd.DataFrame(verification_rows))
else:
    display(pd.DataFrame([{
        "integrity_status": "not_run",
        "reason": "No transcription outputs were generated in this run.",
    }]))


Selected model: openai/whisper-large-v3


,file_path,file_name,extension
0,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\E...,Ecoach LARS interview zelfregie-20260513_16071...,.mp4
1,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,Interview - ervaringen LAge Rug Support-202409...,.mp4
2,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,Interview LAge Rug Support - Hogeschool Rotter...,.mp4
3,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\L...,LARS interview-20240311_100442-Opname van verg...,.mp4
4,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\M...,MTS78-00KV 120224.mp4,.mp4
5,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\R...,RG95-00kv011223.mp4,.mp4


,file_path,exists,is_file,supported_extension,size_bytes,readable,valid,message
0,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\E...,True,True,True,178975668,True,True,OK (ffmpeg-readable media)
1,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,True,True,True,322489051,True,True,OK (ffmpeg-readable media)
2,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,True,True,True,281399464,True,True,OK (ffmpeg-readable media)
3,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\L...,True,True,True,204451411,True,True,OK (ffmpeg-readable media)
4,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\M...,True,True,True,217023984,True,True,OK (ffmpeg-readable media)
5,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\R...,True,True,True,260149165,True,True,OK (ffmpeg-readable media)


Processing 6 file(s)...
[2026-06-06 16:16:37] [INFO] Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.mp4: extracted WAV already exists — reusing Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.extracted.wav
[2026-06-06 16:16:37] [INFO] Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.mp4: using extracted WAV → Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.extracted.wav
[2026-06-06 16:16:37] [INFO] Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.extracted.wav is 40.5 MB — splitting into chunks
[2026-06-06 16:16:37] [INFO] Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.extracted.wav: chunk 1/2 (0.0s–786.0s, 24.0 MB)
[2026-06-06 16:16:50] [INFO] Ecoach LARS interview zelfregie-20260513_160716-Opname van vergadering.extracted.wav: chunk 2/2 (786.0s–1327.8s, 16.5 MB)
[2026-06-06 16:17:07] [INFO] Interview - ervaringen LAge Rug Support-20240902_155757-Op

,file_path,status,detected_language,model_name,text_length,segment_count,transcription_audio,txt_exists,json_exists,csv_exists,xlsx_exists,srt_exists,pdf_exists,integrity_ok,error
0,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\E...,ok,nl,openai/whisper-large-v3,17988,41,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,
1,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,ok,nl,openai/whisper-large-v3,29696,112,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,
2,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\I...,ok,nl,openai/whisper-large-v3,25458,136,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,
3,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\L...,ok,nl,openai/whisper-large-v3,21799,87,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,
4,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\M...,ok,nl,openai/whisper-large-v3,24453,71,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,
5,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\R...,ok,nl,openai/whisper-large-v3,27482,123,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,True,


,file,status,error,txt,txt_exists,json_exists,xlsx_exists,srt_exists,pdf_exists,integrity_ok,txt_bytes,preview
0,Ecoach LARS interview zelfregie-20260513_16071...,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,18051,"Als het goed is, nu netjes aan. De opname gaat..."
1,Interview - ervaringen LAge Rug Support-202409...,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,29726,"Op uw scherm, dat u kan zien dat de opname is ..."
2,Interview LAge Rug Support - Hogeschool Rotter...,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,25490,...pakt. Ja. Hij geeft aan dat hij hem... ...p...
3,LARS interview-20240311_100442-Opname van verg...,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,21837,De opname is nu gestart. We starten met het ge...
4,MTS78-00KV 120224.mp4,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,24476,"Op dit moment. Ja? Ja, prima. Zijn er vragen v..."
5,RG95-00kv011223.mp4,ok,,E:\TEMP\TRANSCRIPTIE_JUNI_2026\0894594@hr.nl\t...,True,True,True,True,True,True,27542,"Oké, helemaal top, helemaal goed. Oké, even ki..."
